# LeetCode 1620 — Coordinate With Maximum Network Quality bằng Python

Link đề: https://leetcode.com/problems/coordinate-with-maximum-network-quality/ · Độ khó: **Medium** · Chủ đề: Array, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def bestCoordinate(self, towers: List[List[int]], radius: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ một bài **mô phỏng / liệt kê** (enumeration): công thức chất lượng tín hiệu `⌊q / (1 + d)⌋`, điều kiện "với tới được" `d ≤ radius`, và luật **hòa thì chọn tọa độ nhỏ nhất theo thứ tự từ điển**.
2. Chứng minh vì sao chỉ cần duyệt lưới `[0, 50] × [0, 50]` (lập luận **"kéo điểm về trong hộp"** — clamp).
3. So sánh khoảng cách bằng **bình phương số nguyên** `dx² + dy² ≤ r²` để tránh sai số số thực; hiểu khi nào `math.sqrt` + `floor` an toàn.
4. Viết nhiều cách: quét lưới rộng, quét lưới `[0,50]²`, **"rải" đóng góp** từ từng tháp lên lưới, và tính chất lượng **thuần số nguyên**.
5. Xây bộ test có seed cố định đối chiếu các cách, kèm các phản ví dụ cho lỗi thường gặp (`<` thay `≤`, sai luật hòa, chỉ xét vị trí tháp…).

## 2. Đề bài & ví dụ

Cho mảng `towers`, mỗi `towers[i] = [x_i, y_i, q_i]` là một trạm phát sóng tại tọa độ nguyên `(x_i, y_i)` với hệ số chất lượng `q_i`. Khoảng cách là **khoảng cách Euclid** `d = √((x − x_i)² + (y − y_i)²)`.

Cho số nguyên `radius`: một trạm **với tới được** từ tọa độ `(x, y)` nếu `d ≤ radius` (**nhỏ hơn hoặc bằng**). Xa hơn thì tín hiệu nhiễu, trạm **không** với tới được.

**Chất lượng tín hiệu** của trạm `i` tại `(x, y)` là `⌊q_i / (1 + d)⌋`. **Chất lượng mạng** tại `(x, y)` là **tổng** chất lượng tín hiệu của mọi trạm **với tới được**.

Trả về `[c_x, c_y]` là tọa độ **nguyên** có chất lượng mạng **lớn nhất**. Nếu nhiều tọa độ cùng chất lượng, trả về tọa độ **không âm** nhỏ nhất theo **thứ tự từ điển** (`(x1, y1) < (x2, y2)` nếu `x1 < x2`, hoặc `x1 == x2` và `y1 < y2`).

**Ví dụ 1:** `towers = [[1,2,5],[2,1,7],[3,1,9]], radius = 2` → `[2,1]`. Tại `(2, 1)` chất lượng là 13:
- trạm `(2,1)`, `q = 7`: `d = 0` ⇒ `⌊7/1⌋ = 7`;
- trạm `(1,2)`, `q = 5`: `d = √2` ⇒ `⌊5/2,414⌋ = ⌊2,07⌋ = 2`;
- trạm `(3,1)`, `q = 9`: `d = 1` ⇒ `⌊9/2⌋ = 4`.

**Ví dụ 2:** `towers = [[23,11,21]], radius = 9` → `[23,11]` (chỉ một trạm, tốt nhất là đứng ngay tại trạm).

**Ví dụ 3:** `towers = [[1,2,13],[2,1,7],[0,1,9]], radius = 2` → `[1,2]`.

**Ràng buộc:**
- `1 ≤ towers.length ≤ 50`, `towers[i].length == 3`
- `0 ≤ x_i, y_i, q_i ≤ 50`
- `1 ≤ radius ≤ 50`

In [1]:
import math
import random
import time
from typing import List

examples = [
    ([[1, 2, 5], [2, 1, 7], [3, 1, 9]], 2, [2, 1]),
    ([[23, 11, 21]], 9, [23, 11]),
    ([[1, 2, 13], [2, 1, 7], [0, 1, 9]], 2, [1, 2]),
]
for towers, r, exp in examples:
    print(f"towers={towers}, radius={r} -> kỳ vọng {exp}")

towers=[[1, 2, 5], [2, 1, 7], [3, 1, 9]], radius=2 -> kỳ vọng [2, 1]
towers=[[23, 11, 21]], radius=9 -> kỳ vọng [23, 11]
towers=[[1, 2, 13], [2, 1, 7], [0, 1, 9]], radius=2 -> kỳ vọng [1, 2]


## 3. Kiến thức nền

### 3.1 Khoảng cách Euclid, `math.sqrt`, `math.hypot`, `math.floor`

`math.sqrt(v)` trả căn bậc hai (số thực), `math.hypot(dx, dy) = √(dx² + dy²)`. `math.floor(v)` là số nguyên lớn nhất `≤ v`. Với số **dương**, `int(v)` cũng cho cùng kết quả (cắt phần thập phân), nhưng với số âm thì khác — ở bài này mọi giá trị đều `≥ 0`.

In [2]:
d = math.hypot(2 - 1, 1 - 2)
print("khoảng cách (1,2)-(2,1):", d, "=", math.sqrt(2))
print("⌊5/(1+√2)⌋ =", math.floor(5 / (1 + d)), "| int(...) =", int(5 / (1 + d)))
print("floor(-2.5) =", math.floor(-2.5), "| int(-2.5) =", int(-2.5))

khoảng cách (1,2)-(2,1): 1.4142135623730951 = 1.4142135623730951
⌊5/(1+√2)⌋ = 2 | int(...) = 2
floor(-2.5) = -3 | int(-2.5) = -2


### 3.2 So sánh khoảng cách bằng bình phương số nguyên

Vì `d ≥ 0` và `radius ≥ 0`: `d ≤ radius ⇔ d² ≤ radius²`. Mà `d² = dx² + dy²` là **số nguyên** ⇒ phép so sánh hoàn toàn chính xác, không cần căn bậc hai.

In [3]:
r = 2
for dx, dy in [(2, 0), (1, 1), (2, 1), (0, 0)]:
    d2 = dx * dx + dy * dy
    print(f"dx={dx}, dy={dy}: d²={d2}, d={math.sqrt(d2):.4f}, với tới được (r={r})? {d2 <= r * r}")

dx=2, dy=0: d²=4, d=2.0000, với tới được (r=2)? True
dx=1, dy=1: d²=2, d=1.4142, với tới được (r=2)? True
dx=2, dy=1: d²=5, d=2.2361, với tới được (r=2)? False
dx=0, dy=0: d²=0, d=0.0000, với tới được (r=2)? True


### 3.3 So sánh theo thứ tự từ điển bằng tuple

Trong Python, `(x1, y1) < (x2, y2)` so sánh **từ điển** đúng như đề định nghĩa. Điều này cho phép chọn tọa độ tốt nhất bằng `max` với khóa `(chất_lượng, −x, −y)`: chất lượng cao hơn thắng; hòa thì `x` nhỏ hơn (tức `−x` lớn hơn) thắng; hòa tiếp thì `y` nhỏ hơn thắng.

In [4]:
print((1, 5) < (2, 0), (2, 0) < (2, 3), (2, 3) < (2, 3))
cands = [(13, 2, 1), (13, 1, 5), (12, 0, 0)]       # (chất lượng, x, y)
best = max(cands, key=lambda t: (t[0], -t[1], -t[2]))
print("tốt nhất:", best)

True True False
tốt nhất: (13, 1, 5)


### 3.4 Thứ tự duyệt lưới và luật hòa

Nếu duyệt `x` tăng dần ở vòng ngoài, `y` tăng dần ở vòng trong, thì các điểm được gặp **đúng theo thứ tự từ điển tăng**. Khi đó chỉ cập nhật đáp án khi chất lượng **lớn hơn hẳn** (`>`), điểm gặp **đầu tiên** trong số các điểm hòa — cũng là điểm nhỏ nhất theo từ điển — được giữ lại.

In [5]:
order = [(x, y) for x in range(3) for y in range(3)]
print(order)
print("đã sắp theo từ điển?", order == sorted(order))

[(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
đã sắp theo từ điển? True


### 3.5 Mảng 2 chiều (lưới) bằng list lồng nhau

`grid = [[0] * 51 for _ in range(51)]` tạo lưới 51×51 toàn số 0. **Không** viết `[[0] * 51] * 51` vì 51 dòng sẽ là **cùng một** list.

In [6]:
good = [[0] * 3 for _ in range(3)]
bad = [[0] * 3] * 3
good[0][0] = 1; bad[0][0] = 1
print("đúng:", good)
print("sai :", bad)

đúng: [[1, 0, 0], [0, 0, 0], [0, 0, 0]]
sai : [[1, 0, 0], [1, 0, 0], [1, 0, 0]]


## 4. Ý tưởng nền tảng / trực giác

Tính bằng tay **Ví dụ 1** (`radius = 2`) tại vài điểm:

| điểm | trạm (1,2,q=5) | trạm (2,1,q=7) | trạm (3,1,q=9) | tổng |
|---|---|---|---|---|
| (2,1) | d=√2 → 2 | d=0 → 7 | d=1 → 4 | **13** |
| (2,2) | d=1 → 2 | d=1 → 3 | d=√2 → 3 | 8 |
| (3,1) | d=√5 > 2 → không | d=1 → 3 | d=0 → 9 | 12 |
| (1,2) | d=0 → 5 | d=√2 → 2 | d=√5 → không | 7 |

**Quan sát:**
1. Đứng **gần** trạm có `q` lớn thì lợi; nhưng phải cân bằng vì có nhiều trạm.
2. Chất lượng chỉ phụ thuộc vào điểm, và mọi thứ đều nhỏ (`≤ 50`) ⇒ cứ **thử hết** các điểm nguyên "hợp lý" là được.
3. Câu hỏi then chốt: **"hợp lý" là những điểm nào?** Lưới vô hạn thì không thử hết được.

In [7]:
def quality_at(towers, radius, x, y):
    total = 0
    for tx, ty, q in towers:
        d = math.sqrt((tx - x) ** 2 + (ty - y) ** 2)
        if d <= radius:
            total += math.floor(q / (1 + d))
    return total

towers, r, _ = examples[0]
for p in [(2, 1), (2, 2), (3, 1), (1, 2), (0, 0)]:
    print(p, "->", quality_at(towers, r, *p))

(2, 1) -> 13
(2, 2) -> 8
(3, 1) -> 12
(1, 2) -> 7
(0, 0) -> 0


## 5. Cách 1 — Brute Force / baseline: quét một lưới rộng `[0, 100] × [0, 100]`

**Ý tưởng:** đáp án phải là tọa độ **không âm**. Một điểm có tọa độ `> 100` cách mọi trạm (tọa độ `≤ 50`) hơn `50 ≥ radius` ⇒ không với tới trạm nào ⇒ chất lượng 0 — không thể hơn `(0, 0)`. Vậy quét `[0, 100]²` là **chắc chắn đủ**, chưa cần suy nghĩ gì sâu.

Duyệt `x` tăng, `y` tăng, cập nhật khi chất lượng **lớn hơn hẳn**; khởi tạo `best = 0`, `ans = [0, 0]` (trường hợp mọi điểm đều 0 thì đáp án là `[0, 0]`).

In [8]:
def best_wide(towers, radius):
    best, ans = 0, [0, 0]
    for x in range(101):
        for y in range(101):
            t = quality_at(towers, radius, x, y)
            if t > best:
                best, ans = t, [x, y]
    return ans

for towers, r, exp in examples:
    t0 = time.perf_counter(); got = best_wide(towers, r); dt = time.perf_counter() - t0
    print(got, "kỳ vọng", exp, f"({dt*1000:.1f} ms)"); assert got == exp

[2, 1] kỳ vọng [2, 1] (4.3 ms)
[23, 11] kỳ vọng [23, 11] (1.5 ms)
[1, 2] kỳ vọng [1, 2] (4.2 ms)


**Độ phức tạp:** `101² · n ≈ 10⁴ · 50 = 5·10⁵` phép tính — vẫn chạy được, nhưng ~75% số điểm là thừa.

**Nút thắt:** (1) quét nhiều điểm chắc chắn không phải đáp án; (2) dùng `sqrt` cho mọi cặp (điểm, trạm), kể cả cặp ở ngoài bán kính.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — "kéo về trong hộp" (clamp):** gọi `H = [0, 50] × [0, 50]` chứa mọi trạm. Với điểm `p = (x, y)` bất kỳ, đặt `c = (clamp(x, 0, 50), clamp(y, 0, 50))`. Vì mọi `x_i ∈ [0, 50]`, ta có `|clamp(x) − x_i| ≤ |x − x_i|` (tương tự với `y`) ⇒ `dist(c, trạm i) ≤ dist(p, trạm i)` với **mọi** `i`. Do đó:
- trạm nào với tới được từ `p` thì cũng với tới được từ `c`;
- `⌊q/(1 + d)⌋` **không giảm** khi `d` giảm.

⇒ `quality(c) ≥ quality(p)`. Nếu `p` có tọa độ `> 50` thì `c` vừa tốt bằng hoặc hơn, vừa **nhỏ hơn** theo từ điển. Nếu `p` có tọa độ âm thì `p` không được chọn (đề yêu cầu không âm). ⇒ **Chỉ cần quét `H = [0, 50]²`** (2601 điểm).

**Insight 2 — so sánh bán kính bằng số nguyên:** `d ≤ r ⇔ dx² + dy² ≤ r²`. Chỉ tính `sqrt` cho các trạm **trong** bán kính.

**Insight 3 — đảo vai "điểm hỏi trạm" thành "trạm rải lên điểm":** thay vì mỗi điểm hỏi 50 trạm, mỗi trạm chỉ "rải" đóng góp lên các điểm trong hình tròn bán kính `r` quanh nó (cắt theo `H`). Khi `r` nhỏ, số phép tính giảm mạnh.

**Quy tắc hòa:** duyệt theo thứ tự từ điển và cập nhật bằng `>` (không phải `≥`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quét `[0, 50]²` + so sánh bán kính bằng số nguyên

**Giải thích từng đoạn:**
1. `r2 = radius * radius`.
2. Với mỗi `(x, y)` trong `[0, 50]²` (duyệt từ điển), với mỗi trạm: `d2 = dx² + dy²`; nếu `d2 ≤ r2` thì cộng `⌊q / (1 + √d2)⌋`.
3. Cập nhật đáp án khi tổng **lớn hơn hẳn**.

In [9]:
def best_grid(towers, radius):
    r2 = radius * radius
    best, ans = 0, [0, 0]
    for x in range(51):
        for y in range(51):
            t = 0
            for tx, ty, q in towers:
                d2 = (tx - x) ** 2 + (ty - y) ** 2
                if d2 <= r2:
                    t += int(q / (1 + math.sqrt(d2)))
            if t > best:
                best, ans = t, [x, y]
    return ans

for towers, r, exp in examples:
    t0 = time.perf_counter(); got = best_grid(towers, r); dt = time.perf_counter() - t0
    print(got, "kỳ vọng", exp, f"({dt*1000:.1f} ms)"); assert got == exp

[2, 1] kỳ vọng [2, 1] (0.7 ms)
[23, 11] kỳ vọng [23, 11] (0.3 ms)
[1, 2] kỳ vọng [1, 2] (0.6 ms)


**Code nộp LeetCode (Cách 2):** xem mục 9 (bản được chọn để nộp).

**Độ phức tạp Cách 2:** Time `O(51² · n) ≈ 1,3·10⁵`, Space `O(1)`.

### 7.2 Cách 3 — "Rải" đóng góp của từng trạm lên lưới

**Giải thích từng đoạn:**
1. Lưới `grid[x][y] = 0` cho `x, y ∈ [0, 50]`.
2. Với mỗi trạm `(tx, ty, q)`: duyệt `x ∈ [max(0, tx − r), min(50, tx + r)]`, `y` tương tự (hình vuông bao hình tròn); nếu `dx² + dy² ≤ r²` thì `grid[x][y] += ⌊q/(1 + d)⌋`.
3. Duyệt lưới theo thứ tự từ điển, lấy ô có giá trị lớn nhất (cập nhật bằng `>`).

Mẹo nhỏ: trạm có `q = 0` không đóng góp gì ⇒ bỏ qua. Hơn nữa `⌊q/(1+d)⌋ = 0` khi `d ≥ q` ⇒ chỉ cần duyệt bán kính `min(r, q)`.

In [10]:
def best_scatter(towers, radius):
    grid = [[0] * 51 for _ in range(51)]
    r2 = radius * radius
    for tx, ty, q in towers:
        R = min(radius, q)                       # xa hơn q thì đóng góp = 0
        for x in range(max(0, tx - R), min(50, tx + R) + 1):
            dx2 = (x - tx) ** 2
            for y in range(max(0, ty - R), min(50, ty + R) + 1):
                d2 = dx2 + (y - ty) ** 2
                if d2 <= r2:
                    grid[x][y] += int(q / (1 + math.sqrt(d2)))
    best, ans = 0, [0, 0]
    for x in range(51):
        for y in range(51):
            if grid[x][y] > best:
                best, ans = grid[x][y], [x, y]
    return ans

for towers, r, exp in examples:
    got = best_scatter(towers, r); print(got, "kỳ vọng", exp); assert got == exp

[2, 1] kỳ vọng [2, 1]
[23, 11] kỳ vọng [23, 11]
[1, 2] kỳ vọng [1, 2]


**Vì sao được cắt bán kính về `min(r, q)`:** nếu `d ≥ q` thì `q/(1 + d) ≤ q/(1 + q) < 1` ⇒ phần nguyên bằng 0. (Với `q = 0` thì `R = 0`, chỉ duyệt đúng ô của trạm và cộng 0.)

**Code nộp LeetCode (Cách 3):**

In [11]:
class SolutionScatter:
    def bestCoordinate(self, towers: List[List[int]], radius: int) -> List[int]:
        grid = [[0] * 51 for _ in range(51)]
        r2 = radius * radius
        for tx, ty, q in towers:
            R = min(radius, q)
            for x in range(max(0, tx - R), min(50, tx + R) + 1):
                for y in range(max(0, ty - R), min(50, ty + R) + 1):
                    d2 = (x - tx) ** 2 + (y - ty) ** 2
                    if d2 <= r2:
                        grid[x][y] += int(q / (1 + math.sqrt(d2)))
        best, ans = 0, [0, 0]
        for x in range(51):
            for y in range(51):
                if grid[x][y] > best:
                    best, ans = grid[x][y], [x, y]
        return ans

print([SolutionScatter().bestCoordinate(t, r) for t, r, _ in examples])

[[2, 1], [23, 11], [1, 2]]


**Độ phức tạp Cách 3:** Time `O(n · min(2R+1, 51)² + 51²)` với `R = min(radius, q)` — tệ nhất vẫn `O(51² · n)` nhưng thường nhỏ hơn nhiều khi `radius` hoặc `q` nhỏ. Space `O(51²)` cho lưới.

### 7.3 Cách 4 — Tính chất lượng thuần số nguyên (không phụ thuộc số thực)

`k = ⌊q / (1 + √d2)⌋` là số nguyên **lớn nhất** thỏa `k · (1 + √d2) ≤ q` ⇔ `k · √d2 ≤ q − k` ⇔ `q − k ≥ 0` **và** `k² · d2 ≤ (q − k)²`. Điều kiện này chỉ dùng phép nhân số nguyên ⇒ **chính xác tuyệt đối**. Vì `q ≤ 50`, ta có thể tìm `k` bằng cách ước lượng rồi chỉnh ±1, hoặc tra bảng tính sẵn cho mọi cặp `(q, d2)`.

**Giải thích từng đoạn:**
1. `ok(k)`: `k ≤ q and k*k*d2 <= (q-k)*(q-k)`.
2. Ước lượng `k = q // (2 + isqrt(d2))` — luôn `≤` đáp án vì `√d2 < isqrt(d2) + 1` nên `2 + isqrt(d2) > 1 + √d2`. (Cẩn thận: `q // (1 + isqrt(d2))` lại là cận **trên**, vì `isqrt(d2) ≤ √d2`!) Rồi tăng `k` chừng nào `ok(k + 1)` — tối đa vài bước.
3. Tra bảng `SIG[q][d2]` cho mọi `q ≤ 50`, `d2 ≤ 2500` (chỉ cần `d2 ≤ r²`).

In [12]:
def signal_int(q, d2):
    """⌊q / (1 + √d2)⌋ chỉ bằng số nguyên."""
    k = q // (2 + math.isqrt(d2))                 # cận dưới: 2 + isqrt(d2) > 1 + sqrt(d2)
    while k + 1 <= q and (k + 1) * (k + 1) * d2 <= (q - k - 1) * (q - k - 1):
        k += 1
    return k

# Đối chiếu với công thức số thực trên toàn bộ miền (q <= 50, d2 <= 2*50^2)
mismatch = [(q, d2) for q in range(51) for d2 in range(5001) if signal_int(q, d2) != math.floor(q / (1 + math.sqrt(d2)))]
print("Số cặp (q, d2) lệch giữa số nguyên và số thực:", len(mismatch))
assert not mismatch
print("Ví dụ: q=9, d2=1 ->", signal_int(9, 1), "| q=5, d2=2 ->", signal_int(5, 2), "| q=12, d2=9 ->", signal_int(12, 9))

Số cặp (q, d2) lệch giữa số nguyên và số thực: 0
Ví dụ: q=9, d2=1 -> 4 | q=5, d2=2 -> 2 | q=12, d2=9 -> 3


In [13]:
def best_integer(towers, radius):
    r2 = radius * radius
    SIG = {}                                      # cache (q, d2) -> tín hiệu
    best, ans = 0, [0, 0]
    for x in range(51):
        for y in range(51):
            t = 0
            for tx, ty, q in towers:
                d2 = (tx - x) ** 2 + (ty - y) ** 2
                if d2 <= r2 and q:
                    key = (q, d2)
                    if key not in SIG:
                        SIG[key] = signal_int(q, d2)
                    t += SIG[key]
            if t > best:
                best, ans = t, [x, y]
    return ans

for towers, r, exp in examples:
    got = best_integer(towers, r); print(got, "kỳ vọng", exp); assert got == exp

[2, 1] kỳ vọng [2, 1]
[23, 11] kỳ vọng [23, 11]
[1, 2] kỳ vọng [1, 2]


**Code nộp LeetCode (Cách 4):**

In [14]:
class SolutionInteger:
    def bestCoordinate(self, towers: List[List[int]], radius: int) -> List[int]:
        def signal(q: int, d2: int) -> int:
            k = q // (2 + math.isqrt(d2))
            while k + 1 <= q and (k + 1) ** 2 * d2 <= (q - k - 1) ** 2:
                k += 1
            return k

        r2 = radius * radius
        best, ans = 0, [0, 0]
        for x in range(51):
            for y in range(51):
                t = 0
                for tx, ty, q in towers:
                    d2 = (tx - x) ** 2 + (ty - y) ** 2
                    if d2 <= r2:
                        t += signal(q, d2)
                if t > best:
                    best, ans = t, [x, y]
        return ans

print([SolutionInteger().bestCoordinate(t, r) for t, r, _ in examples])

[[2, 1], [23, 11], [1, 2]]


**Độ phức tạp Cách 4:** Time `O(51² · n)` (mỗi `signal` là `O(1)` khấu hao — vòng `while` chạy rất ít bước), Space `O(1)` (hoặc `O(#cặp (q, d2))` nếu dùng cache).

> Vì sao Cách 2 dùng số thực vẫn đúng? `√d2` với `d2` là số chính phương được `math.sqrt` trả **chính xác** (chuẩn IEEE-754 làm tròn đúng), nên các trường hợp `q/(1+d)` là số nguyên đều tính đúng; với `d2` không chính phương, `q/(1+√d2)` là số vô tỉ, không bao giờ trùng một số nguyên, và với `q ≤ 50` cũng không đủ gần để sai số làm lệch phần nguyên. Ô code ở trên đã kiểm tra **toàn bộ** miền giá trị: 0 cặp lệch.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 3** (`towers = [[1,2,13],[2,1,7],[0,1,9]]`, `radius = 2`): in ra **mỗi lần đáp án được cập nhật** và chi tiết đóng góp tại các điểm then chốt.

| lần cập nhật | điểm | đóng góp từ (1,2,13) / (2,1,7) / (0,1,9) | chất lượng | best sau đó |
|---|---|---|---|---|
| 1 | (0,0) | ngoài (d²=5) / ngoài (d²=5) / d²=1 → 4 | 4 | 4 |
| 2 | (0,1) | d²=2 → 5 / d²=4 → 2 / d²=0 → 9 | 16 | 16 |
| — | (1,1) | d²=1 → 6 / d²=1 → 3 / d²=1 → 4 | 13 | 16 (không cập nhật) |
| 3 | (1,2) | d²=0 → 13 / d²=2 → 2 / d²=2 → 3 | **18** | 18 |

Mọi điểm sau `(1, 2)` (theo thứ tự từ điển) đều có chất lượng `≤ 18` nên không cập nhật nữa. Hàm `trace_grid` in đúng các lần cập nhật và chi tiết ở các điểm được chọn:

In [15]:
def trace_grid(towers, radius, show_points=()):
    r2 = radius * radius
    best, ans = 0, [0, 0]
    for x in range(51):
        for y in range(51):
            t, parts = 0, []
            for tx, ty, q in towers:
                d2 = (tx - x) ** 2 + (ty - y) ** 2
                if d2 <= r2:
                    s = int(q / (1 + math.sqrt(d2)))
                    t += s; parts.append(f"({tx},{ty},q={q}): d²={d2} -> {s}")
                else:
                    parts.append(f"({tx},{ty},q={q}): d²={d2} > r²={r2} -> bỏ")
            if (x, y) in show_points:
                print(f"  chi tiết tại {(x, y)}: tổng={t}")
                for p in parts:
                    print("     ", p)
            if t > best:
                print(f"cập nhật: ({x},{y}) chất lượng {t} > best {best}")
                best, ans = t, [x, y]
    print("Kết quả:", ans, "chất lượng", best)
    return ans

towers, r, exp = examples[2]
assert trace_grid(towers, r, show_points={(1, 2), (1, 1), (0, 1)}) == exp

cập nhật: (0,0) chất lượng 4 > best 0
  chi tiết tại (0, 1): tổng=16
      (1,2,q=13): d²=2 -> 5
      (2,1,q=7): d²=4 -> 2
      (0,1,q=9): d²=0 -> 9
cập nhật: (0,1) chất lượng 16 > best 4
  chi tiết tại (1, 1): tổng=13
      (1,2,q=13): d²=1 -> 6
      (2,1,q=7): d²=1 -> 3
      (0,1,q=9): d²=1 -> 4
  chi tiết tại (1, 2): tổng=18
      (1,2,q=13): d²=0 -> 13
      (2,1,q=7): d²=2 -> 2
      (0,1,q=9): d²=2 -> 3
cập nhật: (1,2) chất lượng 18 > best 16
Kết quả: [1, 2] chất lượng 18


Đọc output: tại `(1, 2)` trạm `(1,2,q=13)` cho trọn 13, trạm `(2,1)` và `(0,1)` đều cách `√2` cho `⌊7/2,414⌋ = 2` và `⌊9/2,414⌋ = 3` ⇒ tổng **18**. Điểm `(1, 1)` ở "giữa" cả ba trạm nhưng chỉ được `⌊13/2⌋ + ⌊7/2⌋ + ⌊9/2⌋ = 6 + 3 + 4 = 13` — đứng ngay tại trạm mạnh nhất (`q = 13`) lợi hơn. Mẹo: khi tính tay, luôn đối chiếu lại bằng code.

Trace trên **Ví dụ 1** — chỉ in các lần cập nhật:

In [16]:
towers, r, exp = examples[0]
assert trace_grid(towers, r, show_points={(2, 1)}) == exp

cập nhật: (0,1) chất lượng 4 > best 0
cập nhật: (1,1) chất lượng 8 > best 4
  chi tiết tại (2, 1): tổng=13
      (1,2,q=5): d²=2 -> 2
      (2,1,q=7): d²=0 -> 7
      (3,1,q=9): d²=1 -> 4
cập nhật: (2,1) chất lượng 13 > best 8
Kết quả: [2, 1] chất lượng 13


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — quét `[0, 50]²`, so sánh bán kính bằng số nguyên, cập nhật bằng `>` để giữ tọa độ nhỏ nhất theo từ điển.

In [17]:
class Solution:
    def bestCoordinate(self, towers: List[List[int]], radius: int) -> List[int]:
        r2 = radius * radius
        best, ans = 0, [0, 0]
        for x in range(51):
            for y in range(51):
                total = 0
                for tx, ty, q in towers:
                    d2 = (tx - x) * (tx - x) + (ty - y) * (ty - y)
                    if d2 <= r2:
                        total += int(q / (1 + math.sqrt(d2)))
                if total > best:          # '>' giữ điểm nhỏ nhất theo từ điển khi hòa
                    best = total
                    ans = [x, y]
        return ans

for towers, r, exp in examples:
    got = Solution().bestCoordinate(towers, r)
    print(got, exp); assert got == exp

[2, 1] [2, 1]
[23, 11] [23, 11]
[1, 2] [1, 2]


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (clamp).** Với mọi điểm nguyên `p` và `c = clamp(p)` vào `[0, 50]²`: `quality(c) ≥ quality(p)`. *Chứng minh:* với mỗi trạm `i`, `|clamp(x) − x_i| ≤ |x − x_i|` vì `x_i ∈ [0, 50]` (kéo `x` về phía đoạn chứa `x_i` không làm xa thêm); tương tự cho `y` ⇒ `d_i(c) ≤ d_i(p)`. Trạm với tới từ `p` (`d_i(p) ≤ r`) thì cũng với tới từ `c`, và `⌊q_i/(1 + d)⌋` không tăng theo `d` ⇒ từng số hạng của `quality(c)` `≥` số hạng tương ứng của `quality(p)`; các trạm chỉ với tới từ `c` cộng thêm số `≥ 0`. ∎

**Bổ đề 2 (đủ không gian tìm kiếm).** Gọi `M` = chất lượng lớn nhất trên mọi điểm nguyên **không âm**. Theo Bổ đề 1, `M` đạt được trong `[0, 50]²`. Hơn nữa, nếu điểm không âm `p ∉ [0, 50]²` đạt `M`, thì `clamp(p)` cũng đạt `M` và `clamp(p) < p` theo từ điển (ít nhất một tọa độ giảm, và tọa độ `x` không tăng) ⇒ điểm nhỏ nhất theo từ điển trong số các điểm đạt `M` nằm trong `[0, 50]²`.

**Bổ đề 3 (luật hòa).** Duyệt `x` tăng rồi `y` tăng thăm các điểm theo thứ tự từ điển tăng; cập nhật bằng `>` giữ điểm **đầu tiên** đạt giá trị lớn nhất ⇒ đó là điểm nhỏ nhất theo từ điển. Khởi tạo `best = 0, ans = [0, 0]` khớp với trường hợp mọi điểm có chất lượng 0 (đáp án là `(0, 0)` — điểm nhỏ nhất).

**Cách 3** cho đúng cùng giá trị `grid[x][y] = quality(x, y)` (mỗi cặp (điểm, trạm) có đóng góp khác 0 đều được cộng đúng một lần — các cặp bị bỏ có đóng góp 0), nên kết quả trùng Cách 2. **Cách 4** chỉ thay cách tính `⌊q/(1+√d2)⌋` bằng định nghĩa số nguyên tương đương.

## 11. Phân tích Time / Space Complexity

Gọi `n` = số trạm (`≤ 50`), `G = 51` = cạnh lưới, `r` = `radius`, `Q` = `max q`.

| Cách | Time | Space |
|---|---|---|
| 1. Lưới `[0,100]²` | `O(101² · n) ≈ 5·10⁵` | `O(1)` |
| 2. Lưới `[0,50]²` + `d² ≤ r²` | `O(G² · n) ≈ 1,3·10⁵` | `O(1)` |
| 3. Rải đóng góp | `O(n · min(2·min(r, q)+1, G)² + G²)` | `O(G²)` |
| 4. Số nguyên thuần | `O(G² · n)` | `O(1)` |

Không có khác biệt average/worst cho Cách 1, 2, 4 (luôn duyệt toàn lưới). Cách 3 nhanh hẳn khi `radius` hoặc `q` nhỏ. Đo thử:

In [18]:
rng = random.Random(1620)
cases = {
    "r lớn, q lớn": ([[rng.randint(0, 50), rng.randint(0, 50), rng.randint(30, 50)] for _ in range(50)], 50),
    "r nhỏ (=3)":   ([[rng.randint(0, 50), rng.randint(0, 50), rng.randint(0, 50)] for _ in range(50)], 3),
}
for name, (towers, r) in cases.items():
    row = []
    for fname, f in [("C1", best_wide), ("C2", best_grid), ("C3", best_scatter), ("C4", best_integer)]:
        t0 = time.perf_counter(); res = f(towers, r); dt = time.perf_counter() - t0
        row.append(f"{fname}={res} {dt*1000:.0f}ms")
    print(f"{name:<14}", " | ".join(row))

r lớn, q lớn   C1=[33, 15] 68ms | C2=[33, 15] 18ms | C3=[33, 15] 14ms | C4=[33, 15] 25ms


r nhỏ (=3)     C1=[31, 45] 56ms | C2=[31, 45] 9ms | C3=[31, 45] 0ms | C4=[31, 45] 10ms


## 12. Bộ test

Gồm: ví dụ đề, edge case (mọi `q = 0`, trạm ở góc `(0,0)` / `(50,50)`, hai trạm trùng vị trí, hòa giữa nhiều điểm, đúng biên bán kính), và đối chiếu các cách trên nhiều bộ ngẫu nhiên có seed cố định, đúng ràng buộc đề. Cách 1 (chậm nhất) chỉ chạy trên một phần test.

In [19]:
ALL = {
    "grid": best_grid, "scatter": best_scatter, "integer": best_integer,
    "SolScatter": lambda t, r: SolutionScatter().bestCoordinate(t, r),
    "SolInteger": lambda t, r: SolutionInteger().bestCoordinate(t, r),
    "submit": lambda t, r: Solution().bestCoordinate(t, r),
}
def run_all(towers, radius, with_wide=False):
    res = {k: tuple(f(towers, radius)) for k, f in ALL.items()}
    if with_wide:
        res["wide"] = tuple(best_wide(towers, radius))
    assert len(set(res.values())) == 1, (towers, radius, res)
    return list(next(iter(res.values())))

for towers, r, exp in examples:
    assert run_all(towers, r, with_wide=True) == exp

edge = [
    ([[5, 5, 0]], 3, [0, 0]),                          # mọi chất lượng = 0 -> (0,0)
    ([[0, 0, 0], [50, 50, 0]], 50, [0, 0]),
    ([[50, 50, 7]], 1, [50, 50]),                      # trạm ở góc xa
    ([[0, 0, 1]], 50, [0, 0]),
    ([[3, 3, 10], [3, 3, 10]], 5, [3, 3]),             # hai trạm trùng vị trí
    ([[0, 0, 5], [3, 0, 5]], 1, [0, 0]),               # hòa 5 = 5 -> chọn (0,0)
    ([[0, 0, 10], [2, 0, 10], [4, 0, 10]], 2, [2, 0]), # trạm cách đúng bằng radius vẫn tính
    ([[1, 0, 10], [0, 1, 10], [2, 1, 10], [1, 2, 10]], 1, [1, 1]),  # đáp án không phải vị trí trạm
]
for towers, r, exp in edge:
    assert run_all(towers, r, with_wide=True) == exp, (towers, r)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [20]:
rng = random.Random(16200)
for it in range(120):
    n = rng.randint(1, 12)
    span = rng.choice([5, 15, 50])
    towers = [[rng.randint(0, span), rng.randint(0, span), rng.randint(0, 50)] for _ in range(n)]
    r = rng.randint(1, rng.choice([3, 10, 50]))
    run_all(towers, r, with_wide=(it < 10))
print("120 test ngẫu nhiên nhỏ/vừa OK (10 test có cả Cách 1)")

for it in range(8):                                   # đúng giới hạn lớn nhất
    towers = [[rng.randint(0, 50), rng.randint(0, 50), rng.randint(0, 50)] for _ in range(50)]
    run_all(towers, rng.randint(1, 50))
print("8 test n = 50 OK")

120 test ngẫu nhiên nhỏ/vừa OK (10 test có cả Cách 1)


8 test n = 50 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` thay vì `≤` cho bán kính

In [21]:
def wrong_strict(towers, radius):
    best, ans = 0, [0, 0]
    for x in range(51):
        for y in range(51):
            t = 0
            for tx, ty, q in towers:
                d = math.sqrt((tx - x) ** 2 + (ty - y) ** 2)
                if d < radius:                                  # SAI
                    t += int(q / (1 + d))
            if t > best:
                best, ans = t, [x, y]
    return ans

towers, r = [[0, 0, 10], [2, 0, 10], [4, 0, 10]], 2      # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_strict(towers, r), "| đúng:", Solution().bestCoordinate(towers, r), "| brute:", best_wide(towers, r))

sai : [0, 0] | đúng: [2, 0] | brute: [2, 0]


**Vì sao sai:** đề nói "**less than or equal to** radius". Ở `(2, 0)` hai trạm hai bên cách đúng 2 phải được tính: `10 + 3 + 3 = 16`. Bỏ chúng thì mọi điểm chỉ còn 10 và luật hòa chọn `(0, 0)`. **Sửa:** `d2 <= r * r`.

### Lỗi 2 — Cập nhật bằng `>=` (sai luật hòa)

In [22]:
def wrong_tie(towers, radius):
    r2 = radius * radius
    best, ans = 0, [0, 0]
    for x in range(51):
        for y in range(51):
            t = sum(int(q / (1 + math.sqrt((tx - x) ** 2 + (ty - y) ** 2)))
                    for tx, ty, q in towers if (tx - x) ** 2 + (ty - y) ** 2 <= r2)
            if t >= best:                                       # SAI
                best, ans = t, [x, y]
    return ans

towers, r = [[0, 0, 5], [3, 0, 5]], 1
print("sai :", wrong_tie(towers, r), "| đúng:", Solution().bestCoordinate(towers, r))

sai : [3, 0] | đúng: [0, 0]


**Vì sao sai:** với `≥`, điểm **sau cùng** trong các điểm hòa được giữ — tức điểm **lớn nhất** theo từ điển. (Đặc biệt khi mọi chất lượng bằng 0, `≥` trả `(50, 50)` thay vì `(0, 0)`.) **Sửa:** dùng `>` khi duyệt theo thứ tự từ điển tăng.

### Lỗi 3 — Chỉ thử các vị trí của trạm

In [23]:
def wrong_only_towers(towers, radius):
    best, ans = -1, None
    for tx, ty, _ in sorted(towers):
        t = quality_at(towers, radius, tx, ty)
        if t > best:
            best, ans = t, [tx, ty]
    return ans

towers, r = [[1, 0, 10], [0, 1, 10], [2, 1, 10], [1, 2, 10]], 1   # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_only_towers(towers, r), "| đúng:", Solution().bestCoordinate(towers, r))
print("chất lượng tại (0,1):", quality_at(towers, r, 0, 1), "| tại (1,1):", quality_at(towers, r, 1, 1))

sai : [0, 1] | đúng: [1, 1]
chất lượng tại (0,1): 10 | tại (1,1): 20


**Vì sao sai:** điểm tốt nhất có thể nằm **giữa** các trạm. Ở đây `(1, 1)` cách cả 4 trạm đúng 1 ⇒ `4 · ⌊10/2⌋ = 20`, trong khi đứng tại một trạm chỉ được 10 (các trạm khác cách `√2` hoặc 2 > 1). **Sửa:** duyệt toàn lưới `[0, 50]²`.

### Lỗi 4 — Khởi tạo đáp án bằng vị trí trạm đầu tiên

In [24]:
def wrong_init(towers, radius):
    r2 = radius * radius
    best, ans = 0, [towers[0][0], towers[0][1]]                # SAI
    for x in range(51):
        for y in range(51):
            t = sum(int(q / (1 + math.sqrt((tx - x) ** 2 + (ty - y) ** 2)))
                    for tx, ty, q in towers if (tx - x) ** 2 + (ty - y) ** 2 <= r2)
            if t > best:
                best, ans = t, [x, y]
    return ans

towers, r = [[5, 5, 0]], 3
print("sai :", wrong_init(towers, r), "| đúng:", Solution().bestCoordinate(towers, r))

sai : [5, 5] | đúng: [0, 0]


**Vì sao sai:** khi mọi trạm có `q = 0`, mọi điểm có chất lượng 0; không lần cập nhật nào xảy ra nên `ans` giữ giá trị khởi tạo `(5, 5)`, trong khi đáp án đúng là điểm nhỏ nhất `(0, 0)`. **Sửa:** khởi tạo `ans = [0, 0]`, `best = 0` (hoặc `best = −1` để chắc chắn điểm `(0,0)` được ghi nhận).

### Lỗi 5 — Quên căn bậc hai (dùng `d²` trong công thức)

In [25]:
def wrong_no_sqrt(towers, radius):
    r2 = radius * radius
    best, ans = 0, [0, 0]
    for x in range(51):
        for y in range(51):
            t = 0
            for tx, ty, q in towers:
                d2 = (tx - x) ** 2 + (ty - y) ** 2
                if d2 <= r2:
                    t += q // (1 + d2)                                # SAI: phải là 1 + sqrt(d2)
            if t > best:
                best, ans = t, [x, y]
    return ans

towers, r, exp = examples[0]
print("Ví dụ 1 -> sai :", wrong_no_sqrt(towers, r), "| đúng:", exp)
print("chất lượng tại (2,1): sai =", 7 + 5 // 3 + 9 // 2, "| đúng =", quality_at(towers, r, 2, 1))
towers, r = [[2, 0, 5], [0, 0, 4]], 3                                # ví dụ tự dựng (đối chiếu lại đề gốc)
print("Ví dụ tự dựng -> sai :", wrong_no_sqrt(towers, r), "| đúng:", Solution().bestCoordinate(towers, r))

Ví dụ 1 -> sai : [2, 1] | đúng: [2, 1]
chất lượng tại (2,1): sai = 12 | đúng = 13
Ví dụ tự dựng -> sai : [0, 0] | đúng: [2, 0]


**Vì sao sai:** `d` là khoảng cách, không phải bình phương khoảng cách. Dùng `d²` làm tín hiệu suy giảm quá nhanh, nên chất lượng tại `(2,1)` thành 12 thay vì 13. Có khi tọa độ vẫn trùng, nhưng có khi lệch hẳn — ở ví dụ tự dựng, `(2,0)` đáng lẽ thắng nhờ `5 + ⌊4/(1+2)⌋ = 6` (so với `(0,0)`: `4 + ⌊5/3⌋ = 5`), nhưng công thức sai cho `(2,0)`: `5 + 4//5 = 5` và `(0,0)`: `4 + 5//5 = 5` — hòa, nên luật hòa chọn `(0,0)`. **Sửa:** `d2` chỉ dùng để **so sánh bán kính**; công thức tín hiệu cần `√d2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ràng buộc **rất nhỏ** (tọa độ `≤ 50`, `n ≤ 50`) ⇒ nghĩ ngay tới **liệt kê toàn bộ** (enumeration); `51² · 50 ≈ 1,3·10⁵`.
- Câu hỏi "duyệt đến đâu là đủ?" ⇒ lập luận **clamp về hộp bao** các đối tượng.
- So sánh khoảng cách với bán kính ⇒ dùng **bình phương số nguyên**.
- Luật hòa "nhỏ nhất theo từ điển" ⇒ duyệt theo thứ tự từ điển + cập nhật bằng `>`; hoặc dùng khóa tuple `(giá trị, −x, −y)`.
- Hàm suy giảm theo khoảng cách + nhiều nguồn ⇒ cân nhắc **"rải" đóng góp từ nguồn** (đảo vòng lặp) khi bán kính nhỏ.
- Có `floor` của số thực ⇒ tự hỏi có sai số không; nếu nghi ngờ, viết dạng **số nguyên tương đương**.

## 15. Biến thể của bài

1. **Khoảng cách Manhattan / Chebyshev** thay cho Euclid: chỉ đổi công thức `d`; lập luận clamp vẫn đúng.
2. **Tọa độ lớn** (`≤ 10⁹`) và `radius` nhỏ: không quét lưới được; chỉ xét các điểm trong hình tròn quanh từng trạm (Cách 3 với `dict` thay cho lưới).
3. **Nhiều truy vấn bán kính** khác nhau trên cùng bộ trạm — Bài tập 5.
4. **Trả về cả giá trị chất lượng** hoặc **mọi** tọa độ tối ưu — Bài tập 4.
5. Bài liên quan: **1828 Queries on Number of Points Inside a Circle** (đếm điểm trong hình tròn), **149 Max Points on a Line** (liệt kê cặp), **2101 Detonate the Maximum Bombs** (bán kính + đồ thị).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lưới rộng `[0,100]²` | Thử mọi điểm "chắc chắn đủ" | `O(101²·n)` | `O(1)` | Khi chưa chứng minh được phạm vi; kiểm chứng |
| 2. Lưới `[0,50]²` | Clamp + so sánh `d² ≤ r²` | `O(51²·n)` | `O(1)` | **Bản nộp khuyên dùng** |
| 3. Rải đóng góp | Mỗi trạm cộng lên các ô trong bán kính `min(r, q)` | `O(n·min(2R+1,51)² + 51²)` | `O(51²)` | `radius`/`q` nhỏ, hoặc tọa độ lớn (dùng dict) |
| 4. Số nguyên thuần | `k² · d² ≤ (q − k)²` | `O(51²·n)` | `O(1)` | Khi lo sai số số thực, hoặc ngôn ngữ không có `sqrt` chính xác |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Tính `⌊q/(1 + √d2)⌋` cho: (a) `q=10, d2=0`; (b) `q=10, d2=1`; (c) `q=10, d2=4`; (d) `q=10, d2=2`; (e) `q=3, d2=9`.

**Bài 2 (tính tay):** Với **Ví dụ 3**, tính chất lượng tại `(0, 1)` và `(2, 1)`, giải thích vì sao chúng thua `(1, 2)`.

**Bài 3 (điền TODO):** Hoàn thiện `signal(q, d2, radius)` trả tín hiệu của một trạm (0 nếu ngoài bán kính), chỉ dùng số nguyên cho phép so sánh bán kính.

**Bài 4 (tự viết lại):** Viết `all_best(towers, radius)` trả `(chất_lượng_max, danh_sách_mọi_tọa_độ_tối_ưu_trong_[0,50]²)` theo thứ tự từ điển. Kiểm tra phần tử đầu danh sách trùng với `bestCoordinate`.

**Bài 5 (mini challenge):** Viết `best_for_radii(towers, radii)` trả đáp án `bestCoordinate` cho **mỗi** bán kính trong `radii` (tới 50 bán kính) nhanh hơn việc gọi lại lời giải `len(radii)` lần: tiền xử lý cho mỗi điểm danh sách `(d2, tín hiệu)` sắp theo `d2` + tổng tiền tố, rồi với mỗi bán kính dùng `bisect`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def signal_todo(q, d2, radius):
    # TODO: nếu d2 > radius * radius thì trả 0
    # TODO: ngược lại trả int(q / (1 + math.sqrt(d2)))
    pass

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def all_best(towers, radius):
    # TODO: quét lưới, giữ best và list các điểm đạt best (reset list khi best tăng)
    pass

def best_for_radii(towers, radii):
    # TODO: với mỗi điểm: list (d2, sig) sắp theo d2, prefix sum; mỗi radius: bisect_right theo r*r
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `√0 = 0`, `√1 = 1`, `√4 = 2`, `√2 ≈ 1,414`, `√9 = 3`. Mức 2: chia rồi lấy phần nguyên. Mức 3: (a) `10`; (b) `5`; (c) `3`; (d) `⌊10/2,414⌋ = 4`; (e) `⌊3/4⌋ = 0`.

**Bài 2.** Mức 1: tại `(0,1)`: trạm `(0,1)` cho 9, trạm `(1,2)` cách `√2` cho `⌊13/2,414⌋ = 5`, trạm `(2,1)` cách 2 cho `⌊7/3⌋ = 2`. Mức 2: tại `(2,1)`: `7 + ⌊13/2,414⌋ + ⌊9/3⌋ = 7 + 5 + 3`. Mức 3: `(0,1)` = 16, `(2,1)` = 15, đều < 18 của `(1,2)`.

**Bài 3.** Mức 1: tính `radius * radius`. Mức 2: so `d2` với nó. Mức 3: nhớ `≤` chứ không phải `<`.

**Bài 4.** Mức 1: như Cách 2 nhưng thêm `points = []`. Mức 2: `t > best` ⇒ `best = t; points = [(x, y)]`; `t == best` ⇒ `points.append((x, y))`. Mức 3: duyệt từ điển nên `points` tự sắp; `points[0]` chính là đáp án bài gốc.

**Bài 5.** Mức 1: với mỗi điểm `(x, y)`, tạo list `(d2, sig)` cho mọi trạm (sig tính với `√d2`, chưa xét bán kính). Mức 2: sắp theo `d2`, tạo `keys` và `prefix`. Mức 3: với bán kính `r`, `k = bisect_right(keys, r*r)`, chất lượng = `prefix[k]`; rồi chọn điểm tốt nhất theo luật hòa. Đối chiếu với `Solution().bestCoordinate` cho từng `r`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
b1 = [int(q / (1 + math.sqrt(d2))) for q, d2 in [(10, 0), (10, 1), (10, 4), (10, 2), (3, 9)]]
print("Bài 1:", b1); assert b1 == [10, 5, 3, 4, 0]
assert b1 == [signal_int(q, d2) for q, d2 in [(10, 0), (10, 1), (10, 4), (10, 2), (3, 9)]]
towers, r, _ = examples[2]
b2 = {p: quality_at(towers, r, *p) for p in [(0, 1), (2, 1), (1, 2)]}
print("Bài 2:", b2); assert b2 == {(0, 1): 16, (2, 1): 15, (1, 2): 18}

Bài 1: [10, 5, 3, 4, 0]
Bài 2: {(0, 1): 16, (2, 1): 15, (1, 2): 18}


In [29]:
# Bài 3, 4 — đáp án
def signal_todo(q, d2, radius):
    if d2 > radius * radius:
        return 0
    return int(q / (1 + math.sqrt(d2)))

def all_best(towers, radius):
    best, points = -1, []
    for x in range(51):
        for y in range(51):
            t = sum(signal_todo(q, (tx - x) ** 2 + (ty - y) ** 2, radius) for tx, ty, q in towers)
            if t > best:
                best, points = t, [(x, y)]
            elif t == best:
                points.append((x, y))
    return best, points

for towers, r, _ in examples + [([[0, 0, 5], [3, 0, 5]], 1, None)]:
    val, pts = all_best(towers, r)
    print(f"max={val}, {len(pts)} điểm tối ưu, đầu tiên {pts[0]}")
    assert list(pts[0]) == Solution().bestCoordinate(towers, r) and pts == sorted(pts)
print("Bài 3, 4: OK")

max=13, 1 điểm tối ưu, đầu tiên (2, 1)
max=21, 1 điểm tối ưu, đầu tiên (23, 11)
max=18, 1 điểm tối ưu, đầu tiên (1, 2)
max=5, 2 điểm tối ưu, đầu tiên (0, 0)
Bài 3, 4: OK


In [30]:
# Bài 5 — đáp án
from bisect import bisect_right
from itertools import accumulate

def best_for_radii(towers, radii):
    pre = []                                         # theo thứ tự từ điển của điểm
    for x in range(51):
        for y in range(51):
            items = sorted(((tx - x) ** 2 + (ty - y) ** 2, int(q / (1 + math.sqrt((tx - x) ** 2 + (ty - y) ** 2))))
                           for tx, ty, q in towers)
            keys = [d2 for d2, _ in items]
            prefix = [0] + list(accumulate(s for _, s in items))
            pre.append((x, y, keys, prefix))
    out = []
    for r in radii:
        r2 = r * r
        best, ans = 0, [0, 0]
        for x, y, keys, prefix in pre:
            t = prefix[bisect_right(keys, r2)]
            if t > best:
                best, ans = t, [x, y]
        out.append(ans)
    return out

rng = random.Random(7)
towers = [[rng.randint(0, 50), rng.randint(0, 50), rng.randint(0, 50)] for _ in range(20)]
radii = list(range(1, 51, 7))
t0 = time.perf_counter(); fast = best_for_radii(towers, radii); dt = time.perf_counter() - t0
slow = [Solution().bestCoordinate(towers, r) for r in radii]
print("Bài 5:", fast, f"({dt*1000:.0f} ms)")
assert fast == slow
print("Bài 5: khớp với gọi lại lời giải cho từng bán kính")

Bài 5: [[6, 36], [6, 35], [6, 35], [6, 35], [6, 35], [6, 35], [6, 35], [6, 35]] (35 ms)
Bài 5: khớp với gọi lại lời giải cho từng bán kính


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "liệt kê lưới + luật hòa từ điển":**

```python
r2 = radius * radius
best, ans = 0, [0, 0]
for x in range(51):                       # x tăng
    for y in range(51):                   # y tăng  -> thứ tự từ điển
        t = 0
        for tx, ty, q in towers:
            d2 = (tx - x) ** 2 + (ty - y) ** 2
            if d2 <= r2:                  # <=, so sánh số nguyên
                t += int(q / (1 + math.sqrt(d2)))
        if t > best:                      # '>' giữ điểm nhỏ nhất khi hòa
            best, ans = t, [x, y]
return ans
```

**Checklist:**
- [ ] Bán kính dùng `≤` (trạm cách đúng `radius` vẫn được tính)?
- [ ] So sánh bán kính bằng `d² ≤ r²` (số nguyên), công thức tín hiệu dùng `√d²`?
- [ ] Duyệt đủ `[0, 50]²` (không chỉ vị trí trạm, không chỉ hộp bao chặt của các trạm mà không lý giải)?
- [ ] Duyệt theo thứ tự từ điển và cập nhật bằng `>`?
- [ ] Khởi tạo `ans = [0, 0]` cho trường hợp mọi chất lượng bằng 0?
- [ ] Trả về `list` `[x, y]` (không phải tuple)?
- [ ] Đã test: mọi `q = 0`, hòa, đúng biên bán kính, điểm tối ưu nằm giữa các trạm, ngẫu nhiên đối chiếu?